# Lab 6 - PID TempLABUdeA (Notebook Unificado)

Este notebook unifica:
- Cálculo de Kp, Ti, Td por ecuaciones en polo deseado (fsolve).
- Trazado del LGR.
- Ajuste PID por optimización para cumplir %OS, Ts y margen de fase.
- Simulación al escalón y guardado de resultados.

In [ ]:
# Si estas en Colab y faltan paquetes, descomenta esta celda:
# !pip -q install control scipy matplotlib numpy

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import control as ct
from scipy.optimize import fsolve, differential_evolution

# Parametros del laboratorio
K = 0.293
TAU = 169.0
L_DELAY = 12.6

OS_TARGET = 10.0
TS_TARGET = 120.0
PM_TARGET = 45.0

def resolve_base_dir() -> Path:
    cwd = Path.cwd()
    lab6_dir = cwd / 'Lab 6'
    return lab6_dir if lab6_dir.exists() else cwd

BASE_DIR = resolve_base_dir()
FIG_DIR = BASE_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR =', BASE_DIR)
print('FIG_DIR  =', FIG_DIR)

## 1) Cálculo por polo deseado (fsolve)

In [ ]:
# Polo dominante deseado desde %OS = 10% y Ts = 120 s
sd = -0.0333 + 0.0455j

def sistema(vars, K, tau, sd):
    kp_test, ti_test, td_test = vars
    g_sd = K / (tau * sd + 1)
    gpid_sd = kp_test * (1 + 1/(ti_test * sd) + td_test * sd)
    l_sd = gpid_sd * g_sd

    err = l_sd + 1
    eq1 = np.real(err)
    eq2 = np.imag(err)
    eq3 = td_test - (ti_test / 4.0)
    return [eq1, eq2, eq3]

x0 = [100.0, TAU, TAU / 4.0]
sol, info, ier, msg = fsolve(lambda x: sistema(x, K, TAU, sd), x0, full_output=True)

if ier != 1:
    print('fsolve no convergio:', msg)
else:
    kp_f, ti_f, td_f = sol
    print(f'Kp (fsolve) = {kp_f:.6f}')
    print(f'Ti (fsolve) = {ti_f:.6f} s')
    print(f'Td (fsolve) = {td_f:.6f} s')

    with open(BASE_DIR / 'compute_pid_results.json', 'w', encoding='utf-8') as f:
        json.dump({
            'K': K,
            'tau': TAU,
            'L_delay': L_DELAY,
            'sd_real': float(np.real(sd)),
            'sd_imag': float(np.imag(sd)),
            'Kp': float(kp_f),
            'Ti_s': float(ti_f),
            'Td_s': float(td_f)
        }, f, indent=2)
    print('Guardado:', BASE_DIR / 'compute_pid_results.json')

## 2) LGR + simulación al escalón con ajuste automático

In [ ]:
def pid_tf(kp, ti, td):
    s = ct.tf([1, 0], [1])
    return kp * (1 + 1/(ti*s) + td*s)

def plant_with_delay(k, tau, delay):
    g = ct.tf([k], [tau, 1])
    num_d, den_d = ct.pade(delay, 1)
    d = ct.tf(num_d, den_d)
    return g * d

def settling_time_2pct(t, y, yss):
    tol = 0.02 * abs(yss)
    idx = np.where(np.abs(y - yss) > tol)[0]
    if idx.size == 0:
        return 0.0
    if idx[-1] == len(t) - 1:
        return float(t[-1])
    return float(t[idx[-1] + 1])

def eval_metrics(kp, ti, td):
    g = plant_with_delay(K, TAU, L_DELAY)
    c = pid_tf(kp, ti, td)
    l_open = c * g
    t_closed = ct.feedback(l_open, 1)

    poles = ct.poles(t_closed)
    stable = np.all(np.real(poles) < 0)

    t = np.linspace(0, 1200, 6000)
    t, y = ct.step_response(t_closed, t)

    yss = float(np.real(y[-1]))
    ymax = float(np.max(np.real(y)))

    if not np.isfinite(yss) or abs(yss) < 1e-6:
        return {
            'stable': False,
            'os': 1e6,
            'ts': 1e6,
            'pm': -1e6,
            'yss': yss,
            't': t,
            'y': y,
            'L': l_open,
            'T': t_closed,
        }

    os = max(0.0, (ymax - yss) / abs(yss) * 100.0)
    ts = settling_time_2pct(t, np.real(y), yss)

    gm, pm, wcg, wcp = ct.margin(l_open)
    if pm is None or np.isnan(pm):
        pm = -180.0

    return {
        'stable': bool(stable),
        'os': float(os),
        'ts': float(ts),
        'pm': float(pm),
        'yss': yss,
        't': t,
        'y': np.real(y),
        'L': l_open,
        'T': t_closed,
    }

def objective(x):
    kp = 10 ** x[0]
    ti = 10 ** x[1]
    td = 10 ** x[2]

    m = eval_metrics(kp, ti, td)

    penalty = 200.0 if not m['stable'] else 0.0
    if not np.isfinite(m['yss']):
        return 1e6

    ess = min(abs(1.0 - m['yss']) * 100.0, 1e4)
    os_val = min(max(m['os'], 0.0), 1e4)
    ts_val = min(max(m['ts'], 0.0), 1e4)
    pm_val = m['pm'] if np.isfinite(m['pm']) else -180.0

    j_os = ((os_val - OS_TARGET) / 10.0) ** 2
    j_ts = ((ts_val - TS_TARGET) / 80.0) ** 2
    j_pm = ((max(0.0, PM_TARGET - pm_val)) / 10.0) ** 2
    j_ess = (ess / 2.0) ** 2
    return j_os + j_ts + j_pm + j_ess + penalty

warnings.filterwarnings(
    'ignore',
    message='Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.'
)
warnings.filterwarnings(
    'ignore',
    message='Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.'
)

# LGR con Ti=tau, Td=tau/10
ti_guess = TAU
td_guess = TAU / 10.0
s = ct.tf([1, 0], [1])
lgr_open = K * (ti_guess * td_guess * s**2 + ti_guess * s + 1) / (ti_guess * s * (TAU * s + 1))

plt.figure(figsize=(8, 6))
ct.root_locus_plot(lgr_open, grid=True)
plt.title('Lugar Geometrico de las Raices (Ti=tau, Td=tau/10)')
plt.xlabel('Parte real')
plt.ylabel('Parte imaginaria')
plt.tight_layout()
plt.savefig(FIG_DIR / 'lgr_lab6.png', dpi=160)
plt.show()

# Optimizacion PID
bounds = [(-2.0, 4.0), (0.0, 3.5), (-1.0, 3.0)]
result = differential_evolution(
    objective,
    bounds=bounds,
    maxiter=45,
    popsize=12,
    seed=7,
    polish=True,
    tol=1e-3
)

kp = 10 ** result.x[0]
ti = 10 ** result.x[1]
td = 10 ** result.x[2]
m = eval_metrics(kp, ti, td)

t = m['t']
y = m['y']
yss = m['yss']
os_limit = yss * (1 + OS_TARGET / 100.0)

plt.figure(figsize=(8, 6))
plt.plot(t, y, 'b', linewidth=1.8, label='Respuesta simulada')
plt.axhline(os_limit, color='r', linestyle='--', linewidth=1.3, label='Limite de sobreimpulso (10%)')
plt.axvline(m['ts'], color='g', linestyle='--', linewidth=1.3, label=f"Ts (2%) = {m['ts']:.2f} s")
plt.title('Respuesta al escalon en lazo cerrado')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')
plt.grid(True, alpha=0.3)
plt.legend(loc='best')
plt.tight_layout()
plt.savefig(FIG_DIR / 'step_lab6.png', dpi=160)
plt.show()

out = {
    'K': K,
    'tau': TAU,
    'L_delay': L_DELAY,
    'Kp': float(kp),
    'Ti': float(ti),
    'Td': float(td),
    'OS_percent': float(m['os']),
    'Ts_2pct_s': float(m['ts']),
    'PM_deg': float(m['pm']),
    'yss': float(yss),
    'stable': bool(m['stable'])
}

with open(BASE_DIR / 'simulation_results.json', 'w', encoding='utf-8') as f:
    json.dump(out, f, indent=2)

print('=== Resultados de simulacion Lab 6 ===')
for k, v in out.items():
    print(f'{k}: {v}')
print('Figura LGR :', FIG_DIR / 'lgr_lab6.png')
print('Figura Step:', FIG_DIR / 'step_lab6.png')
print('JSON      :', BASE_DIR / 'simulation_results.json')